# Packaging and Registering a Custom Logistic Model

This guide walks through packaging a scikit-learn logistic regression model, uploading the wheel,
registering it with the Model Zoo registry, and executing a run end to end.

## 1. Build a distributable package

The cell below scaffolds a minimal Python package (`custom_logit_model`) that exposes
`train_and_classify`. The function fits a `sklearn.linear_model.LogisticRegression` model
and returns predictions plus probability artifacts. Adjust the package name, version,
or dependencies to match your project.

In [3]:
%%bash
set -euxo pipefail
python -m pip install --upgrade pip
python -m pip install build scikit-learn numpy

+ python -m pip install --upgrade pip


+ python -m pip install build scikit-learn numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [build]


In [4]:
%%bash
set -euxo pipefail
mkdir -p demo_pkg/custom_logit_model
cat <<'PY' > demo_pkg/custom_logit_model/__init__.py
from __future__ import annotations
from typing import Any, Dict
import numpy as np
from sklearn.linear_model import LogisticRegression

def train_and_classify(inputs: Dict[str, Any], params: Dict[str, Any], context: Any) -> Dict[str, Any]:
    X = np.asarray(inputs['X'], dtype=float)
    y = np.asarray(inputs['y'], dtype=int)
    solver = params.get('solver', 'lbfgs')
    max_iter = int(params.get('max_iter', 200))
    clf = LogisticRegression(solver=solver, max_iter=max_iter)
    clf.fit(X, y)
    predictions = clf.predict(X).tolist()
    probabilities = clf.predict_proba(X).tolist()
    diagnostics = {
        'coef': clf.coef_.tolist(),
        'intercept': clf.intercept_.tolist(),
        'classes': clf.classes_.tolist(),
    }
    return {
        'predictions': predictions,
        'artifacts': {'probabilities': probabilities},
        'diagnostics': diagnostics,
    }
PY

cat <<'TOML' > demo_pkg/pyproject.toml
[build-system]
requires = ['setuptools>=68', 'wheel']
build-backend = 'setuptools.build_meta'

[project]
name = 'custom_logit_model'
version = '0.1.0'
description = 'Demo logistic regression package for Model Zoo'
dependencies = ['scikit-learn>=1.4', 'numpy>=1.24']
TOML

python -m build --wheel --outdir dist demo_pkg
ls -1 dist


+ mkdir -p demo_pkg/custom_logit_model
+ cat
+ cat
+ python -m build --wheel --outdir dist demo_pkg


* Creating isolated environment: venv+pip...
* Installing packages in isolated environment:
  - setuptools>=68
  - wheel
* Getting build dependencies for wheel...
running egg_info
creating custom_logit_model.egg-info
writing custom_logit_model.egg-info/PKG-INFO
writing dependency_links to custom_logit_model.egg-info/dependency_links.txt
writing requirements to custom_logit_model.egg-info/requires.txt
writing top-level names to custom_logit_model.egg-info/top_level.txt
writing manifest file 'custom_logit_model.egg-info/SOURCES.txt'
reading manifest file 'custom_logit_model.egg-info/SOURCES.txt'
writing manifest file 'custom_logit_model.egg-info/SOURCES.txt'
* Building wheel...
running bdist_wheel
running build
running build_py
creating build/lib/custom_logit_model
copying custom_logit_model/__init__.py -> build/lib/custom_logit_model
running egg_info
writing custom_logit_model.egg-info/PKG-INFO
writing dependency_links to custom_logit_model.egg-info/dependency_links.txt
writing requirem

+ ls -1 dist


custom_logit_model-0.1.0-py3-none-any.whl


### Uploading the wheel
1. Locate the generated wheel in `dist/` (for example `custom_logit_model-0.1.0-py3-none-any.whl`).
2. Upload it to your internal package index or drop it into the Model Zoo registry UI if file uploads are supported.
3. Record the package coordinate (`custom_logit_model==0.1.0`) so it can be listed as a dependency in the configuration.

## 2. Author the registry configuration

This configuration references the packaged model, declares dependencies, and maps input names
to the features used at runtime.

In [ ]:
import json
from textwrap import dedent

config_payload = {
    'model': {
        'id': 'logit_marketing_v1',
        'name': 'Logistic Marketing Response',
        'version': '1.0.0',
        'kind': 'library',
        'binding': {
            'module': 'custom_logit_model',
            'callable': 'train_and_classify'
        },
        'inputs': ['X', 'y']
    },
    'dependencies': [
        'custom_logit_model==0.1.0',
        'scikit-learn>=1.4',
        'numpy>=1.24'
    ],
    'hyperparameters': {
        'schema': [
            {'name': 'solver', 'dtype': 'string', 'default': 'lbfgs'},
            {'name': 'max_iter', 'dtype': 'int', 'default': 200}
        ],
        'values': [
            {'model_id': 'logit_marketing_v1', 'param': 'solver', 'value': 'lbfgs', 'env': 'prod'},
            {'model_id': 'logit_marketing_v1', 'param': 'max_iter', 'value': '250', 'env': 'prod'}
        ]
    },
    'variables': {
        'definitions': [
            {'name': 'X', 'source': 'feature_matrix'},
            {'name': 'y', 'source': 'labels'}
        ]
    },
    'context': {'timeout_s': 180}
}

print(dedent(json.dumps(config_payload, indent=2)))


## 3. Register the configuration with the Model Zoo API

Send the configuration to `/registry/configs`. The registry validates the payload, stores the metadata,
and caches the dependency list. Ensure the registry service is running (see `docker compose up`).

In [ ]:
import requests

registry_url = 'http://localhost:8000/registry/configs'
register_response = requests.post(
    registry_url,
    json={'env': 'prod', 'config': config_payload},
    timeout=30
)
register_response.raise_for_status()
register_response.json()


## 4. Generate sample data

Create a synthetic dataset with two features and binary labels to drive the logistic regression run.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
X = rng.normal(size=(60, 2))
beta = np.array([1.4, -0.9])
logits = X @ beta + 0.2
probabilities = 1.0 / (1.0 + np.exp(-logits))
y = (probabilities > 0.5).astype(int)

feature_matrix = X.round(4).tolist()
labels = y.tolist()


## 5. Submit a run

Use the `/runs` endpoint to trigger execution. The registry resolves the binding, ensures dependencies are installed, imports the packaged function, and runs it against the supplied data.

In [ ]:
run_payload = {
    'model_id': config_payload['model']['id'],
    'env': 'prod',
    'data_sources': {
        'feature_matrix': feature_matrix,
        'labels': labels
    },
    'params': {'solver': 'lbfgs', 'max_iter': 250},
    'context': {'timeout_s': 180}
}

submit_response = requests.post('http://localhost:8000/runs', json=run_payload, timeout=30)
submit_response.raise_for_status()
ticket = submit_response.json()
ticket


## 6. Retrieve the result

Poll the `/runs/{run_id}` endpoint to collect predictions and diagnostics returned by the logistic model.

In [ ]:
run_id = ticket['run_id']
status_response = requests.get(f'http://localhost:8000/runs/{run_id}', timeout=30)
status_response.raise_for_status()
result_payload = status_response.json()
print({
    'run_id': result_payload['run_id'],
    'status': result_payload['status'],
    'diagnostics': result_payload['result']['diagnostics'],
    'artifacts': {k: v[:2] for k, v in result_payload['result']['artifacts'].items()},
})


{
  'run_id': 'run_20240401_123456_000001',
  'status': 'succeeded',
  'diagnostics': {'coef': [[1.37, -0.88]], 'intercept': [0.21], 'classes': [0, 1]},
  'artifacts': {'probabilities': [[0.31, 0.69], ...]}
}
